# Same-Doctor, Same-#ABL Analysis

The core question is:

**If we hold the doctor and the number of ablation lesions (`#ABL`) constant, how much variation is still left, where does it appear, and does that pattern fit what we know from the transcript?**

This is a stronger design for your case because it separates:

- **between-group effects**: different doctors and different lesion counts
- **within-group variation**: cases that should look similar on observable workload but still take different amounts of time

That remaining within-group variation is the part most likely to reflect hidden operational differences such as communication, verification, coordination, setup, and short waits between steps.


## Hypothesis

We will test three claims:

1. `doctor` and `#ABL` explain some variation in procedure time, but not all of it.
2. Even within the same doctor and the same `#ABL`, there is still meaningful variation across cases.
3. The remaining variation should be concentrated in coordination-heavy phases that line up with the transcript, especially prep, transseptal work, and non-energy parts of ablation.

If this pattern appears, then the transcript can be used as an interpretation layer for the hidden operational variability.


## Important Assumption

The transcript is **not** from one of the timed cases in the dataset. It is a representative description of the standard procedure.

That means we use the transcript as a **process map**, not as direct evidence for any specific case. The timed dataset tells us **where variation exists**. The transcript helps us describe **what kinds of coded micro-steps are embedded inside those macro stages and which hidden-variability categories those micro-steps belong to**.


## Coding Approach

To make the representative transcript usable in analysis, we **coded** it at the micro-step level.

- `Micro-step`: a discrete task or event needed for the procedure to move forward
- `Primary category`: the main hidden-variability category assigned to that micro-step

The primary categories used in this notebook are:

- `Communication patterns`
- `Personnel interactions`
- `Equipment handling`
- `Room setup`
- `Procedural manipulation`
- `Imaging and mapping`
- `Verification and safety`
- `Handoff or waiting`

So these labels are not raw fields from the timing dataset. They are the result of a structured LLM-assisted coding of the representative transcript.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from scripts.data_utils import (
    DATA_PATH,
    load_case_data,
)
from scripts.doctor_abl_variation_analysis import (
    build_case_time_ancova,
    build_doctor_abl_cell_summary,
    build_phase_effect_anova,
    build_within_cell_variation_summary,
)
from scripts.transcript_macro_analysis import (
    build_transcript_micro_step_table,
    build_transcript_phase_summary,
)

CASE_TIME = "CASE TIME (Cath In-Out)"
SKIN_SKIN = "SKIN-SKIN (Access to Cath-Out)"
PT_PREP = "PT PREP/INTUBATION Pt-In-Access"
ACCESS = "ACCESSS (Min)"
TSP = "TSP (Min)"
PRE_MAP = "PRE-MAP (Min)"
ABL_DURATION = "ABL DURATION (Abl Start-End)"
ABL_TIME = "ABL TIME (Min)"
POST_CARE = "POST CARE/EXTUBATION (Cath-Out to Pt-Out)"
ABL_COUNT = "#ABL"


## Step 1. Load the data and define the comparison set

We use only **standard cases** for the main analysis. Cases with notes such as `CTI`, `BOX`, `PST BOX`, `SVC`, and `TROUBLESHOOT` are excluded because they represent visibly different work.

We then focus on doctor-by-`#ABL` cells with at least **2 cases**, because those are the cells where within-group variation can actually be measured.


In [2]:
all_cases = load_case_data(DATA_PATH)
standard_cases = all_cases.loc[~all_cases["complex_case"]].copy()

cell_counts = (
    standard_cases.groupby(["PHYSICIAN", ABL_COUNT])
    .size()
    .reset_index(name="n_cases")
    .sort_values(["PHYSICIAN", ABL_COUNT])
)

eligible_cells = cell_counts.loc[cell_counts["n_cases"] >= 2].copy()
cases_in_eligible_cells = int(eligible_cells["n_cases"].sum())

display(cell_counts)
print(f"Total cases in workbook: {len(all_cases)}")
print(f"Standard cases used in this analysis: {len(standard_cases)}")
print(f"Doctor-#ABL cells with at least 2 cases: {len(eligible_cells)}")
print(f"Standard cases covered by repeated doctor-#ABL cells: {cases_in_eligible_cells}")
print(f"Coverage of standard cases: {cases_in_eligible_cells / len(standard_cases):.1%}")


,PHYSICIAN,#ABL,n_cases
0,Dr. A,16.0,1
1,Dr. A,17.0,5
2,Dr. A,18.0,7
3,Dr. A,19.0,17
4,Dr. A,20.0,14
5,Dr. A,21.0,9
6,Dr. A,22.0,2
7,Dr. A,24.0,1
8,Dr. B,15.0,1
9,Dr. B,17.0,1


Total cases in workbook: 150
Standard cases used in this analysis: 126
Doctor-#ABL cells with at least 2 cases: 20
Standard cases covered by repeated doctor-#ABL cells: 116
Coverage of standard cases: 92.1%


### Interpretation of Step 1

This is enough structure to support a same-doctor, same-`#ABL` analysis.

The key point is coverage: if most standard cases fall into repeated doctor-`#ABL` cells, then we can study within-cell variation without throwing away the dataset. That makes the grouped design a practical main analysis rather than a niche subset check.


## Step 2. Use ANOVA/ANCOVA to see which effects matter overall

Before looking inside the cells, we first ask:

- Does `doctor` matter?
- Does `#ABL` matter?
- Where do those effects show up across the procedure?

This does **not** replace the grouped analysis. It only tells us which observable effects are important at the overall dataset level.


In [3]:
phase_effects = build_phase_effect_anova(standard_cases)
case_time_ancova = build_case_time_ancova(standard_cases)

display(case_time_ancova)

phase_view = (
    phase_effects[["outcome", "effect", "F", "p_value", "eta_sq_total"]]
    .sort_values(["outcome", "effect"])
    .reset_index(drop=True)
)
display(phase_view)


,effect,n,r_squared,sum_sq,df,F,p_value,eta_sq_total
0,C(PHYSICIAN),122,0.2658,2185.8069,2.0,8.6312,0.000319,0.1249
1,"Q(""#ABL"")",122,0.2658,354.3231,1.0,2.7983,0.097038,0.0203
2,"Q(""ABL TIME (Min)"")",122,0.2658,139.4731,1.0,1.1015,0.296102,0.0080


,outcome,effect,F,p_value,eta_sq_total
0,ABL DURATION (Abl Start-End),C(PHYSICIAN),7.3340,0.000995,0.0955
1,ABL DURATION (Abl Start-End),"Q(""#ABL"")",20.8500,0.000012,0.1358
2,ACCESSS (Min),C(PHYSICIAN),1.1809,0.310619,0.0195
3,ACCESSS (Min),"Q(""#ABL"")",0.6987,0.404920,0.0058
4,CASE TIME (Cath In-Out),C(PHYSICIAN),9.6360,0.000133,0.1348
5,CASE TIME (Cath In-Out),"Q(""#ABL"")",5.7101,0.018452,0.0399
6,POST CARE/EXTUBATION (Cath-Out to Pt-Out),C(PHYSICIAN),8.9257,0.000247,0.1276
7,POST CARE/EXTUBATION (Cath-Out to Pt-Out),"Q(""#ABL"")",5.0778,0.026097,0.0363
8,PRE-MAP (Min),C(PHYSICIAN),5.2763,0.006386,0.0778
9,PRE-MAP (Min),"Q(""#ABL"")",7.0892,0.008837,0.0523


In [4]:
ancova = case_time_ancova.set_index("effect")

effect_text = f"""
### Interpretation of Step 2

The overall effect tests show that **doctor matters** for case time even after controlling for `#ABL` and `ABL TIME`.

- In the case-time ANCOVA, `doctor` is significant with **p = {ancova.loc['C(PHYSICIAN)', 'p_value']:.6f}**.
- In that same model, `#ABL` is weaker once `ABL TIME` is also included (**p = {ancova.loc['Q("#ABL")', 'p_value']:.6f}**).
- `ABL TIME` itself is not significant in that case-time model (**p = {ancova.loc['Q("ABL TIME (Min)")', 'p_value']:.6f}**).

At the phase level, the pattern is more informative:

- `doctor` matters for **case time, skin-skin, prep/intubation, TSP, PRE-MAP, ablation duration, and post-care**.
- `doctor` does **not** matter much for **access**.
- `#ABL` matters most for **ablation duration**, and it also shows up in **case time, PRE-MAP, and post-care**.

So ANOVA/ANCOVA gives a useful answer to “which effects matter where?”, but it does not answer the main operational question by itself. For that, we need to look at the variation that remains **inside** doctor-by-`#ABL` cells.
"""

display(Markdown(effect_text))



### Interpretation of Step 2

The overall effect tests show that **doctor matters** for case time even after controlling for `#ABL` and `ABL TIME`.

- In the case-time ANCOVA, `doctor` is significant with **p = 0.000319**.
- In that same model, `#ABL` is weaker once `ABL TIME` is also included (**p = 0.097038**).
- `ABL TIME` itself is not significant in that case-time model (**p = 0.296102**).

At the phase level, the pattern is more informative:

- `doctor` matters for **case time, skin-skin, prep/intubation, TSP, PRE-MAP, ablation duration, and post-care**.
- `doctor` does **not** matter much for **access**.
- `#ABL` matters most for **ablation duration**, and it also shows up in **case time, PRE-MAP, and post-care**.

So ANOVA/ANCOVA gives a useful answer to “which effects matter where?”, but it does not answer the main operational question by itself. For that, we need to look at the variation that remains **inside** doctor-by-`#ABL` cells.


## Step 3. Make same-doctor, same-#ABL the main comparison

Now we switch to the main design.

For each doctor and each `#ABL` value, we look at all repeated cases in that cell and measure how much they still vary. This tells us how unstable the process is **after holding doctor and lesion count fixed**.


In [5]:
cell_summary = build_doctor_abl_cell_summary(standard_cases, min_cell_size=2)
within_cell_summary = build_within_cell_variation_summary(cell_summary)

display(cell_summary[[
    "physician",
    "num_abl",
    "cell_n",
    "case_time_cath_in_out_mean",
    "case_time_cath_in_out_sd",
    "case_time_cath_in_out_range",
    "tsp_min_sd",
    "pre_map_min_sd",
    "abl_duration_abl_start_end_sd",
    "abl_time_min_sd",
    "post_care_extubation_cath_out_to_pt_out_sd",
]].sort_values(["physician", "num_abl"]))

display(within_cell_summary)


,physician,num_abl,cell_n,case_time_cath_in_out_mean,case_time_cath_in_out_sd,case_time_cath_in_out_range,tsp_min_sd,pre_map_min_sd,abl_duration_abl_start_end_sd,abl_time_min_sd,post_care_extubation_cath_out_to_pt_out_sd
0,Dr. A,17,5,34.40,9.71,23.0,5.98,0.45,4.49,0.00,2.70
1,Dr. A,18,7,31.14,7.10,22.0,2.29,0.53,5.10,0.11,2.88
2,Dr. A,19,17,31.12,6.49,23.0,2.41,0.47,3.28,0.18,2.63
3,Dr. A,20,14,32.21,5.92,20.0,0.83,0.63,4.24,0.00,2.49
4,Dr. A,21,9,32.67,8.28,27.0,2.64,0.33,4.36,0.22,3.06
5,Dr. A,22,2,32.00,4.24,6.0,2.12,0.00,1.41,0.00,2.12
6,Dr. B,18,2,44.00,8.49,12.0,2.83,0.71,0.71,0.00,3.54
7,Dr. B,19,12,37.67,9.73,32.0,2.91,0.79,4.76,0.14,3.59
8,Dr. B,20,5,34.80,11.84,32.0,1.95,0.55,3.42,0.00,3.03
9,Dr. B,21,9,49.56,18.13,59.0,10.70,1.67,6.84,0.12,4.24


,phase,mean_within_cell_sd,median_within_cell_sd,mean_within_cell_range,median_within_cell_range
0,CASE TIME (Cath In-Out),10.11,7.77,23.55,21.0
1,SKIN-SKIN (Access to Cath-Out),9.96,8.33,22.95,21.5
2,PT PREP/INTUBATION Pt-In-Access,4.19,3.71,10.10,10.5
3,ABL DURATION (Abl Start-End),4.14,4.24,10.30,11.0
4,TSP (Min),2.99,2.04,7.45,4.0
5,POST CARE/EXTUBATION (Cath-Out to Pt-Out),2.95,2.67,6.75,6.5
6,ACCESSS (Min),1.71,1.12,4.05,2.5
7,PRE-MAP (Min),0.60,0.58,1.35,1.0
8,ABL TIME (Min),0.13,0.00,0.27,0.0


In [6]:
w = within_cell_summary.set_index("phase")

grouped_text = f"""
### Interpretation of Step 3

This is the strongest evidence in the notebook.

Even after fixing **doctor** and **`#ABL`**, the process is still quite variable:

- Mean within-cell SD for **case time** = **{w.loc['CASE TIME (Cath In-Out)', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **skin-skin** = **{w.loc['SKIN-SKIN (Access to Cath-Out)', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **prep/intubation** = **{w.loc['PT PREP/INTUBATION Pt-In-Access', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **ablation duration** = **{w.loc['ABL DURATION (Abl Start-End)', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **TSP** = **{w.loc['TSP (Min)', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **ABL TIME** = **{w.loc['ABL TIME (Min)', 'mean_within_cell_sd']:.2f} min**

The contrast is the key result:

- `ABL TIME` is almost fixed inside these cells.
- The larger variation shows up in the **surrounding process**, especially skin-skin, prep, ablation duration, and TSP.

That means the residual variation is not mainly “how much ablation was delivered.” It is much more about **how the case flowed**.
"""

display(Markdown(grouped_text))



### Interpretation of Step 3

This is the strongest evidence in the notebook.

Even after fixing **doctor** and **`#ABL`**, the process is still quite variable:

- Mean within-cell SD for **case time** = **10.11 min**
- Mean within-cell SD for **skin-skin** = **9.96 min**
- Mean within-cell SD for **prep/intubation** = **4.19 min**
- Mean within-cell SD for **ablation duration** = **4.14 min**
- Mean within-cell SD for **TSP** = **2.99 min**
- Mean within-cell SD for **ABL TIME** = **0.13 min**

The contrast is the key result:

- `ABL TIME` is almost fixed inside these cells.
- The larger variation shows up in the **surrounding process**, especially skin-skin, prep, ablation duration, and TSP.

That means the residual variation is not mainly “how much ablation was delivered.” It is much more about **how the case flowed**.


## Step 4. Look at the hidden-time proxies directly

To make the interpretation cleaner, we add three derived measures:

- `ablation_non_energy = ABL DURATION - ABL TIME`
- `post_ablation_la_time = LA DWELL TIME - ABL DURATION`
- `non_procedural_room_time = PT IN-OUT - SKIN-SKIN`

These help separate active treatment from the time around treatment.


In [7]:
extended = standard_cases.copy()
extended["ablation_non_energy"] = extended[ABL_DURATION] - extended[ABL_TIME]
extended["post_ablation_la_time"] = extended["LA DWELL TIME (Abl Start-Cath-Out)"] - extended[ABL_DURATION]
extended["non_procedural_room_time"] = extended["PT IN-OUT (Min)"] - extended[SKIN_SKIN]

metrics = [
    CASE_TIME,
    SKIN_SKIN,
    PT_PREP,
    ACCESS,
    TSP,
    PRE_MAP,
    ABL_DURATION,
    ABL_TIME,
    "ablation_non_energy",
    "post_ablation_la_time",
    POST_CARE,
    "non_procedural_room_time",
]

rows = []
for (physician, abl_count), group in extended.groupby(["PHYSICIAN", ABL_COUNT]):
    if len(group) < 2:
        continue
    row = {"physician": physician, "num_abl": abl_count, "cell_n": len(group)}
    for metric in metrics:
        row[f"{metric}_sd"] = group[metric].dropna().std(ddof=1)
    rows.append(row)

extended_cell = pd.DataFrame(rows)
hidden_variation = pd.DataFrame(
    {
        "metric": metrics,
        "mean_within_cell_sd": [extended_cell[f"{m}_sd"].mean() for m in metrics],
        "median_within_cell_sd": [extended_cell[f"{m}_sd"].median() for m in metrics],
    }
).sort_values("mean_within_cell_sd", ascending=False)

display(hidden_variation.round(2))


,metric,mean_within_cell_sd,median_within_cell_sd
0,CASE TIME (Cath In-Out),10.11,7.78
1,SKIN-SKIN (Access to Cath-Out),9.95,8.33
11,non_procedural_room_time,6.25,5.72
2,PT PREP/INTUBATION Pt-In-Access,4.19,3.71
8,ablation_non_energy,4.14,4.24
6,ABL DURATION (Abl Start-End),4.14,4.24
4,TSP (Min),2.99,2.04
10,POST CARE/EXTUBATION (Cath-Out to Pt-Out),2.95,2.67
3,ACCESSS (Min),1.71,1.13
9,post_ablation_la_time,1.47,0.90


In [8]:
hv = hidden_variation.set_index("metric")

hidden_text = f"""
### Interpretation of Step 4

The hidden-time proxies sharpen the story:

- Mean within-cell SD for **case time** = **{hv.loc['CASE TIME (Cath In-Out)', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **skin-skin** = **{hv.loc['SKIN-SKIN (Access to Cath-Out)', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **non-procedural room time** = **{hv.loc['non_procedural_room_time', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **ablation non-energy time** = **{hv.loc['ablation_non_energy', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **post-ablation LA time** = **{hv.loc['post_ablation_la_time', 'mean_within_cell_sd']:.2f} min**
- Mean within-cell SD for **ABL TIME** = **{hv.loc['ABL TIME (Min)', 'mean_within_cell_sd']:.2f} min**

The important contrast is that **ablation non-energy time varies a lot, while ABL TIME barely varies**. That is exactly the pattern we would expect if hidden variation comes from repositioning, confirmation, communication, setup, and short delays between actions rather than from energy delivery itself.
"""

display(Markdown(hidden_text))



### Interpretation of Step 4

The hidden-time proxies sharpen the story:

- Mean within-cell SD for **case time** = **10.11 min**
- Mean within-cell SD for **skin-skin** = **9.95 min**
- Mean within-cell SD for **non-procedural room time** = **6.25 min**
- Mean within-cell SD for **ablation non-energy time** = **4.14 min**
- Mean within-cell SD for **post-ablation LA time** = **1.47 min**
- Mean within-cell SD for **ABL TIME** = **0.13 min**

The important contrast is that **ablation non-energy time varies a lot, while ABL TIME barely varies**. That is exactly the pattern we would expect if hidden variation comes from repositioning, confirmation, communication, setup, and short delays between actions rather than from energy delivery itself.


## Step 5. Connect the variation pattern to the transcript

Now that the timestamped transcript is available, we can code each macro phase directly.

For each visible phase, we code the transcript to record:

- transcript start and end time
- elapsed transcript duration
- number of timestamp blocks
- coded micro-steps
- coded category counts for the hidden-variability taxonomy

This gives us a transcript-side complexity summary that can be compared directly to the data-side residual variation.


In [9]:
transcript_summary = build_transcript_phase_summary()

transcript_compare = transcript_summary.merge(
    hidden_variation[["metric", "mean_within_cell_sd", "median_within_cell_sd"]],
    left_on="linked_metric",
    right_on="metric",
    how="left",
).drop(columns=["metric"])

display(transcript_compare[[
    "phase_group",
    "transcript_start",
    "transcript_end",
    "duration_min",
    "timestamp_blocks",
    "coded_micro_steps",
    "coded_unique_categories",
    "micro_steps_communication_patterns",
    "micro_steps_personnel_interactions",
    "micro_steps_equipment_handling",
    "micro_steps_room_setup",
    "micro_steps_procedural_manipulation",
    "micro_steps_imaging_and_mapping",
    "micro_steps_verification_and_safety",
    "micro_steps_handoff_or_waiting",
    "mean_within_cell_sd",
    "examples",
]])


,phase_group,transcript_start,transcript_end,duration_min,timestamp_blocks,coded_micro_steps,coded_unique_categories,micro_steps_communication_patterns,micro_steps_personnel_interactions,micro_steps_equipment_handling,micro_steps_room_setup,micro_steps_procedural_manipulation,micro_steps_imaging_and_mapping,micro_steps_verification_and_safety,micro_steps_handoff_or_waiting,mean_within_cell_sd,examples
0,Prep/intubation,00:00:00,00:01:05,1.08,31,11,7,3,1,1,2,0,1,2,1,4.185409,"nurse instructions, checklist, anesthesia prep..."
1,Access/setup,00:01:05,00:03:18,2.22,61,18,5,0,0,7,1,3,1,6,0,1.714453,"ultrasound access, cable setup, irrigation/mec..."
2,TSP,00:03:18,00:07:09,3.85,104,20,5,0,0,3,0,8,4,4,1,2.985421,"sheath flushes, ICE guidance, septal tenting, ..."
3,Pre-map,00:07:09,00:08:02,0.88,23,7,4,0,2,0,0,1,3,0,1,0.601417,"3D geometry, CT comparison, mapping experts, s..."
4,Ablation/non-energy work,00:08:02,00:09:45,1.72,45,9,5,1,0,0,0,2,3,2,1,4.139527,"point-by-point lesion work, multi-screen guida..."
5,Post-care/extubation,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,2.945093,not visible in the transcript excerpt


In [10]:
transcript_step_map = build_transcript_micro_step_table()

display(transcript_step_map)


,phase_group,linked_metric,micro_step,primary_category
0,Prep/intubation,PT PREP/INTUBATION Pt-In-Access,patient positioned on table,Room setup
1,Prep/intubation,PT PREP/INTUBATION Pt-In-Access,nurse final instructions,Communication patterns
2,Prep/intubation,PT PREP/INTUBATION Pt-In-Access,checklist run-through,Communication patterns
3,Prep/intubation,PT PREP/INTUBATION Pt-In-Access,anesthesia readiness,Personnel interactions
4,Prep/intubation,PT PREP/INTUBATION Pt-In-Access,physician safety and goal briefing,Communication patterns
...,...,...,...,...
60,Ablation/non-energy work,ablation_non_energy,continuous live electrogram guidance,Imaging and mapping
61,Ablation/non-energy work,ablation_non_energy,integrate modalities simultaneously,Communication patterns
62,Ablation/non-energy work,ablation_non_energy,add inter-vein lesions when needed,Procedural manipulation
63,Ablation/non-energy work,ablation_non_energy,review chamber voltage pattern,Verification and safety


### Interpretation of the step map

This table shows the actual **micro-step to macro-step mapping** used in the notebook, along with the primary hidden-variability category assigned to each micro-step.

That matters because it makes the transcript contribution explicit:

- `Prep/intubation` includes communication-heavy and verification-heavy micro-steps.
- `Access/setup` is dominated by equipment handling, room setup, and safety checks.
- `TSP` is dominated by procedural manipulation, imaging, and verification.
- `Pre-map` is dominated by imaging/mapping plus personnel interaction.
- `Ablation/non-energy work` mixes procedural manipulation, imaging, and verification around the lesion-delivery sequence.

So when we later compare transcript complexity to measured variability, we are doing it with an explicit coded micro-step map and category taxonomy rather than a vague description.


In [11]:
phase_variability_for_categories = transcript_compare.loc[
    transcript_compare["duration_min"].notna(),
    ["phase_group", "mean_within_cell_sd"],
].copy()

category_summary = (
    transcript_step_map.merge(
        phase_variability_for_categories,
        on="phase_group",
        how="left",
    )
    .groupby("primary_category")
    .agg(
        coded_micro_steps=("micro_step", "count"),
        phases_covered=("phase_group", "nunique"),
        avg_phase_variability=("mean_within_cell_sd", "mean"),
        weighted_variability_exposure=("mean_within_cell_sd", "sum"),
    )
    .sort_values("weighted_variability_exposure", ascending=False)
    .reset_index()
)

display(category_summary.round(2))


,primary_category,coded_micro_steps,phases_covered,avg_phase_variability,weighted_variability_exposure
0,Verification and safety,14,4,2.78,38.88
1,Procedural manipulation,14,4,2.71,37.91
2,Imaging and mapping,12,5,2.67,32.06
3,Equipment handling,11,3,2.29,25.14
4,Communication patterns,4,2,4.17,16.70
5,Handoff or waiting,4,4,2.98,11.91
6,Room setup,3,2,3.36,10.09
7,Personnel interactions,3,2,1.80,5.39


### Interpretation of the category analysis

This is the first place where the new coding scheme becomes analytically useful.

The category table should be read in two ways:

- `coded_micro_steps` tells us how common that category is in the representative procedure
- `avg_phase_variability` and `weighted_variability_exposure` tell us how much that category tends to sit inside macro phases that remain variable in the timed data

So this is not just a transcript count. It is a bridge between the transcript coding and the measured variability.


In [12]:
observed_map = transcript_compare.loc[transcript_compare["duration_min"].notna()].copy()

transcript_text = f"""
### Interpretation of Step 5

The transcript coding fits the data pattern reasonably well.

- **Prep/intubation** lasts **{observed_map.loc[observed_map['phase_group'] == 'Prep/intubation', 'duration_min'].iloc[0]:.2f} transcript minutes** and contains **{int(observed_map.loc[observed_map['phase_group'] == 'Prep/intubation', 'coded_micro_steps'].iloc[0])} coded micro-steps**. Its coded mix is communication-heavy and verification-heavy, and in the timed data it still shows substantial within-cell variation (**{observed_map.loc[observed_map['phase_group'] == 'Prep/intubation', 'mean_within_cell_sd'].iloc[0]:.2f} min**).
- **TSP** is the densest transcript stage: **{observed_map.loc[observed_map['phase_group'] == 'TSP', 'duration_min'].iloc[0]:.2f} minutes**, **{int(observed_map.loc[observed_map['phase_group'] == 'TSP', 'timestamp_blocks'].iloc[0])} timestamp blocks**, and **{int(observed_map.loc[observed_map['phase_group'] == 'TSP', 'coded_micro_steps'].iloc[0])} coded micro-steps**. It is dominated by procedural manipulation, imaging/mapping, and verification/safety, and it also shows meaningful within-cell variation in the data (**{observed_map.loc[observed_map['phase_group'] == 'TSP', 'mean_within_cell_sd'].iloc[0]:.2f} min**).
- **Ablation/non-energy work** contains **{int(observed_map.loc[observed_map['phase_group'] == 'Ablation/non-energy work', 'coded_micro_steps'].iloc[0])} coded micro-steps** and remains highly variable in the data (**{observed_map.loc[observed_map['phase_group'] == 'Ablation/non-energy work', 'mean_within_cell_sd'].iloc[0]:.2f} min**). Its coded categories center on procedural manipulation, imaging/mapping, and verification, which fits the idea that the time around energy delivery is operationally dense.
- **Pre-map** shows real transcript complexity (**{int(observed_map.loc[observed_map['phase_group'] == 'Pre-map', 'coded_micro_steps'].iloc[0])} coded micro-steps**) but relatively low residual variation (**{observed_map.loc[observed_map['phase_group'] == 'Pre-map', 'mean_within_cell_sd'].iloc[0]:.2f} min**). That is a useful nuance: a phase can be complex but still standardized.
- **Access/setup** has many coded micro-steps, especially equipment handling and room setup, but the timed `ACCESS` column is comparatively stable (**{observed_map.loc[observed_map['phase_group'] == 'Access/setup', 'mean_within_cell_sd'].iloc[0]:.2f} min**). That suggests some setup burden is either routinized or absorbed into neighboring timing windows rather than showing up as large residual spread in the access column alone.

This is a good fit to the theory:

The phases with the clearest residual variation after controlling for doctor and `#ABL` are also the phases where the transcript shows dense clusters of coded micro-steps in categories like communication, procedural manipulation, imaging/mapping, and verification/safety. That does not prove second-by-second causality, but it gives a much stronger operational explanation for the remaining variance.
"""

display(Markdown(transcript_text))



### Interpretation of Step 5

The transcript coding fits the data pattern reasonably well.

- **Prep/intubation** lasts **1.08 transcript minutes** and contains **11 coded micro-steps**. Its coded mix is communication-heavy and verification-heavy, and in the timed data it still shows substantial within-cell variation (**4.19 min**).
- **TSP** is the densest transcript stage: **3.85 minutes**, **104 timestamp blocks**, and **20 coded micro-steps**. It is dominated by procedural manipulation, imaging/mapping, and verification/safety, and it also shows meaningful within-cell variation in the data (**2.99 min**).
- **Ablation/non-energy work** contains **9 coded micro-steps** and remains highly variable in the data (**4.14 min**). Its coded categories center on procedural manipulation, imaging/mapping, and verification, which fits the idea that the time around energy delivery is operationally dense.
- **Pre-map** shows real transcript complexity (**7 coded micro-steps**) but relatively low residual variation (**0.60 min**). That is a useful nuance: a phase can be complex but still standardized.
- **Access/setup** has many coded micro-steps, especially equipment handling and room setup, but the timed `ACCESS` column is comparatively stable (**1.71 min**). That suggests some setup burden is either routinized or absorbed into neighboring timing windows rather than showing up as large residual spread in the access column alone.

This is a good fit to the theory:

The phases with the clearest residual variation after controlling for doctor and `#ABL` are also the phases where the transcript shows dense clusters of coded micro-steps in categories like communication, procedural manipulation, imaging/mapping, and verification/safety. That does not prove second-by-second causality, but it gives a much stronger operational explanation for the remaining variance.


In [13]:
cat = category_summary.set_index("primary_category")

category_text = f"""
### What the category results suggest

- **Verification and safety** has the largest weighted variability exposure (**{cat.loc['Verification and safety', 'weighted_variability_exposure']:.2f}**) and one of the highest total coded counts (**{int(cat.loc['Verification and safety', 'coded_micro_steps'])} micro-steps**). That fits the idea that confirmation and safety checks are embedded in the stages where the process still varies.
- **Procedural manipulation** is similarly high (**{cat.loc['Procedural manipulation', 'weighted_variability_exposure']:.2f}**), which makes sense because TSP and ablation non-energy work both require repeated positioning, advancement, and site-to-site movement.
- **Imaging and mapping** is also prominent (**{cat.loc['Imaging and mapping', 'weighted_variability_exposure']:.2f}**), which supports the idea that visualization and interpretation work are tightly linked to variable procedural flow.
- **Communication patterns** has a smaller raw count (**{int(cat.loc['Communication patterns', 'coded_micro_steps'])} micro-steps**) but the **highest average phase variability** (**{cat.loc['Communication patterns', 'avg_phase_variability']:.2f} min**). That means communication-heavy micro-steps are concentrated in some of the most variable phases even if they are not the most numerous overall.
- **Equipment handling** is common (**{int(cat.loc['Equipment handling', 'coded_micro_steps'])} micro-steps**) but less strongly tied to high-variability phases than the categories above. That matches the earlier phase result that access/setup is complex but relatively stable.
- **Personnel interactions** has the lowest average variability exposure (**{cat.loc['Personnel interactions', 'avg_phase_variability']:.2f} min**), suggesting that not all social coordination creates large timing variation when the task is routinized.

So the strongest explanatory categories for variability are not just “more work” in general. They are the categories that combine repeated manipulation, imaging, and safety confirmation inside already sensitive phases like TSP and the non-energy portion of ablation.
"""

display(Markdown(category_text))



### What the category results suggest

- **Verification and safety** has the largest weighted variability exposure (**38.88**) and one of the highest total coded counts (**14 micro-steps**). That fits the idea that confirmation and safety checks are embedded in the stages where the process still varies.
- **Procedural manipulation** is similarly high (**37.91**), which makes sense because TSP and ablation non-energy work both require repeated positioning, advancement, and site-to-site movement.
- **Imaging and mapping** is also prominent (**32.06**), which supports the idea that visualization and interpretation work are tightly linked to variable procedural flow.
- **Communication patterns** has a smaller raw count (**4 micro-steps**) but the **highest average phase variability** (**4.17 min**). That means communication-heavy micro-steps are concentrated in some of the most variable phases even if they are not the most numerous overall.
- **Equipment handling** is common (**11 micro-steps**) but less strongly tied to high-variability phases than the categories above. That matches the earlier phase result that access/setup is complex but relatively stable.
- **Personnel interactions** has the lowest average variability exposure (**1.80 min**), suggesting that not all social coordination creates large timing variation when the task is routinized.

So the strongest explanatory categories for variability are not just “more work” in general. They are the categories that combine repeated manipulation, imaging, and safety confirmation inside already sensitive phases like TSP and the non-energy portion of ablation.


In [14]:
complexity_view = observed_map[[
    "phase_group",
    "coded_micro_steps",
    "coded_unique_categories",
    "timestamp_blocks",
    "mean_within_cell_sd",
]].copy()

complexity_view["variation_rank"] = complexity_view["mean_within_cell_sd"].rank(ascending=False, method="min")
complexity_view["micro_step_rank"] = complexity_view["coded_micro_steps"].rank(ascending=False, method="min")
complexity_view["category_diversity_rank"] = complexity_view["coded_unique_categories"].rank(ascending=False, method="min")

display(complexity_view.sort_values("variation_rank"))


,phase_group,coded_micro_steps,coded_unique_categories,timestamp_blocks,mean_within_cell_sd,variation_rank,micro_step_rank,category_diversity_rank
0,Prep/intubation,11,7,31,4.185409,1.0,3.0,1.0
4,Ablation/non-energy work,9,5,45,4.139527,2.0,4.0,2.0
2,TSP,20,5,104,2.985421,3.0,1.0,2.0
1,Access/setup,18,5,61,1.714453,4.0,2.0,2.0
3,Pre-map,7,4,23,0.601417,5.0,5.0,5.0


### Interpretation of the complexity-versus-variability comparison

This comparison gives a useful but nuanced result.

- The relationship is **not perfectly linear**.
- `TSP`, `Prep/intubation`, and `Ablation/non-energy work` support the theory well: they contain many coded micro-steps across multiple hidden-variability categories, and they also show meaningful residual variation.
- `Pre-map` is short and comparatively stable, which also makes sense.
- `Access/setup` is the main exception: it looks complex in the transcript, but the timed `ACCESS` metric itself is relatively stable. That suggests some setup work is either highly standardized or absorbed into neighboring timing windows rather than appearing as variability in the access column alone.

So the transcript-coding idea is a **solid start**, but it should be interpreted as a structured explanation of likely variability sources, not as a strict rule that “more coded micro-steps or more category diversity always means more timing variation.”


## Final conclusion

For this case, **same-doctor, same-`#ABL` should be the main analysis**.

Why:

- ANOVA/ANCOVA tells us that `doctor` and `#ABL` do matter, but they do not explain everything.
- The grouped analysis shows that even after holding those effects fixed, there is still large variation in case time and skin-skin time.
- That remaining variation is concentrated in prep, TSP, and non-energy ablation time, not in `ABL TIME`.
- The transcript-informed coding fits that pattern reasonably well: the phases with the densest micro-step structure and the richest mix of hidden-variability categories are often the same phases with the strongest residual variation, although the relationship is not perfectly one-to-one.

So the recommended storyline for the report is:

1. Use ANOVA/ANCOVA to establish which observable effects matter overall.
2. Use same-doctor, same-`#ABL` cells as the main analysis of residual variation.
3. Use the transcript to interpret that residual variation as hidden operational variability, especially around communication, confirmation, navigation, and setup.
